In [2]:
import logging

import pandas as pd
import spectrum_fundamentals.constants as c
from spectrum_fundamentals.mod_string import internal_without_mods, maxquant_to_internal

from spectrum_io.search_result.search_results import SearchResults
#from .search_results import SearchResults
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [53]:

logger = logging.getLogger(__name__)


class MaxQuant(SearchResults):
    """Handle search results from MaxQuant."""

    @staticmethod
    def add_tmt_mod(mass: float, seq: str, unimod_tag: str) -> float:
        """
        Add tmt modification.

        :param mass: mass without tmt modification
        :param seq: sequence of the peptide
        :param unimod_tag: UNIMOD tag for the modification
        :return: mass as float
        """
        num_of_tmt = seq.count(unimod_tag)
        mass += num_of_tmt * c.MOD_MASSES[f"{unimod_tag}"]
        return mass

    @staticmethod
    def read_result(path: str, tmt_labeled: str) -> pd.DataFrame:
        """
        Function to read a msms txt and perform some basic formatting.

        :param path: path to msms.txt to read
        :param tmt_labeled: tmt label as str
        :return: pd.DataFrame with the formatted data
        """
        logger.info("Reading msms.txt file")
        df = pd.read_csv(
            path,
            usecols=lambda x: x.upper()
            in [
                "RAW FILE",
                "SCAN NUMBER",
                "MODIFIED SEQUENCE",
                "CHARGE",
                "FRAGMENTATION",
                "MASS ANALYZER",
                "SCAN EVENT NUMBER",
                "LABELING STATE",
                "MASS",  # = Calculated Precursor mass; TODO get column with experimental Precursor mass instead
                "SCORE",
                "REVERSE",
                "RETENTION TIME",
            ],
            sep="\t",
        )
        logger.info("Finished reading msms.txt file")

        # Standardize column names
        df.columns = df.columns.str.upper()
        df.columns = df.columns.str.replace(" ", "_")

        df = MaxQuant.update_columns_for_prosit(df, tmt_labeled)
        df = MaxQuant.filter_valid_prosit_sequences(df)

        return df

    @staticmethod
    def update_columns_for_prosit(df: pd.DataFrame, tmt_labeled: str) -> pd.DataFrame:
        """
        Update columns of df to work with Prosit.

        :param df: df to modify
        :param tmt_labeled: True if tmt labeled
        :return: modified df as pd.DataFrame
        """
        df.rename(columns={"CHARGE": "PRECURSOR_CHARGE"}, inplace=True)

        if "MASS_ANALYZER" not in df.columns:
            df["MASS_ANALYZER"] = "FTMS"
        if "FRAGMENTATION" not in df.columns:
            df["FRAGMENTATION"] = "HCD"
        df["REVERSE"].fillna(False, inplace=True)
        df["REVERSE"].replace("+", True, inplace=True)
        logger.info("Converting MaxQuant peptide sequence to internal format")
        if tmt_labeled != "":
            unimod_tag = c.TMT_MODS[tmt_labeled]
            logger.info("Adding TMT fixed modifications")
            #print( df["MODIFIED_SEQUENCE"])
            df["MODIFIED_SEQUENCE"] = maxquant_to_internal(
                df["MODIFIED_SEQUENCE"].to_numpy(),
                fixed_mods={"C": "C[UNIMOD:4]", "^_": f"_{unimod_tag}", "K": f"K{unimod_tag}"},
            )
            display(df)
            df["MASS"] = df.apply(lambda x: MaxQuant.add_tmt_mod(x.MASS, x.MODIFIED_SEQUENCE, unimod_tag), axis=1)
            if "msa" in tmt_labeled:
                logger.info("Replacing phospho by dehydration for Phospho-MSA")
                df["MODIFIED_SEQUENCE_MSA"] = df["MODIFIED_SEQUENCE"].str.replace(
                    "[UNIMOD:21]", "[UNIMOD:23]", regex=False
                )
        elif "LABELING_STATE" in df.columns:
            logger.info("Adding SILAC fixed modifications")
            df.loc[df["LABELING_STATE"] == 1, "MODIFIED_SEQUENCE"] = maxquant_to_internal(
                df[df["LABELING_STATE"] == 1]["MODIFIED_SEQUENCE"].to_numpy(),
                fixed_mods={"C": "C[UNIMOD:4]", "K": "K[UNIMOD:259]", "R": "R[UNIMOD:267]"},
            )
            df.loc[df["LABELING_STATE"] != 1, "MODIFIED_SEQUENCE"] = maxquant_to_internal(
                df[df["LABELING_STATE"] != 1]["MODIFIED_SEQUENCE"].to_numpy()
            )
            df["MASS"] = df.apply(lambda x: MaxQuant.add_tmt_mod(x.MASS, x.MODIFIED_SEQUENCE, "[UNIMOD:259]"), axis=1)
            df["MASS"] = df.apply(lambda x: MaxQuant.add_tmt_mod(x.MASS, x.MODIFIED_SEQUENCE, "[UNIMOD:267]"), axis=1)
            df.drop(columns=["LABELING_STATE"], inplace=True)
        else:
            df["MODIFIED_SEQUENCE"] = maxquant_to_internal(df["MODIFIED_SEQUENCE"].to_numpy())
        df["SEQUENCE"] = internal_without_mods(df["MODIFIED_SEQUENCE"])
        df["PEPTIDE_LENGTH"] = df["SEQUENCE"].apply(lambda x: len(x))

        return df

    @staticmethod
    def filter_valid_prosit_sequences(df: pd.DataFrame) -> pd.DataFrame:
        """
        Filter valid Prosit sequences.

        :param df: df to filter
        :return: df after filtration
        """
        logger.info(f"#sequences before filtering for valid prosit sequences: {len(df.index)}")
        df = df[(df["PEPTIDE_LENGTH"] <= 30)]
        df = df[(~df["MODIFIED_SEQUENCE"].str.contains("(ac)", regex=False))]
        df = df[(~df["MODIFIED_SEQUENCE"].str.contains("(Acetyl (Protein N-term))", regex=False))]
        df = df[(~df["SEQUENCE"].str.contains("U"))]
        df = df[df["PRECURSOR_CHARGE"] <= 6]
        df = df[df["PEPTIDE_LENGTH"] >= 7]
        logger.info(f"#sequences after filtering for valid prosit sequences: {len(df.index)}")

        return df


In [54]:

path = "/home/mkalhor/wilhelmlab/notebooks/msms.txt"
MaxQuant.read_result(path,tmt_labeled="itraq4")




,RAW_FILE,SCAN_NUMBER,MODIFIED_SEQUENCE,PRECURSOR_CHARGE,FRAGMENTATION,MASS_ANALYZER,SCAN_EVENT_NUMBER,MASS,RETENTION_TIME,SCORE,REVERSE
0,Plamsa_03401_RD2_30min_Top10_86ms_RP13_R1,917,[UNIMOD:214]AAAATVEEPQVGIVGR,3,HCD,FTMS,5,1566.8366,1.9775,3.4214,True
1,Plamsa_03401_RD2_30min_Top10_86ms_RP13_R1,18153,[UNIMOD:214]AAADLMTYC[UNIMOD:4]DAHAC[UNIMOD:4]...,4,HCD,FTMS,4,3618.6429,30.0850,1.6419,False
2,Plamsa_03401_RD2_30min_Top10_86ms_RP13_R1,9832,[UNIMOD:214]AAC[UNIMOD:4]GPSSC[UNIMOD:4]YALFPR,2,HCD,FTMS,7,1555.6912,16.2500,163.6400,False
3,Plamsa_03401_RD2_30min_Top10_86ms_RP13_R1,4680,[UNIMOD:214]AAC[UNIMOD:4]LLPK[UNIMOD:214],2,HCD,FTMS,3,771.4313,7.9360,91.0930,False
4,Plamsa_03401_RD2_30min_Top10_86ms_RP13_R1,2418,[UNIMOD:214]AAC[UNIMOD:4]LRRC[UNIMOD:4]K[UNIMO...,2,HCD,FTMS,4,1033.5273,4.3071,17.2700,False
...,...,...,...,...,...,...,...,...,...,...,...
15503,Plamsa_03401_RD2_30min_Top10_86ms_RP13_R1,7640,[UNIMOD:214]YYQTDLVNIMK[UNIMOD:214],3,HCD,FTMS,7,1386.6853,12.7810,15.5390,False
15504,Plamsa_03401_RD2_30min_Top10_86ms_RP13_R1,10582,[UNIMOD:214]YYSNVC[UNIMOD:4]QR,2,HCD,FTMS,9,1088.4709,17.4690,17.3860,False
15505,Plamsa_03401_RD2_30min_Top10_86ms_RP13_R1,15474,[UNIMOD:214]YYTALYRK[UNIMOD:214]MLDPGLMTC[UNIM...,3,HCD,FTMS,6,2310.1211,25.5650,8.8569,False
15506,Plamsa_03401_RD2_30min_Top10_86ms_RP13_R1,12042,[UNIMOD:214]YYVESC[UNIMOD:4]LDQPILQVDK[UNIMOD:...,3,HCD,FTMS,9,1968.9503,19.8670,7.7095,True


,RAW_FILE,SCAN_NUMBER,MODIFIED_SEQUENCE,PRECURSOR_CHARGE,FRAGMENTATION,MASS_ANALYZER,SCAN_EVENT_NUMBER,MASS,RETENTION_TIME,SCORE,REVERSE,SEQUENCE,PEPTIDE_LENGTH
0,Plamsa_03401_RD2_30min_Top10_86ms_RP13_R1,917,[UNIMOD:214]AAAATVEEPQVGIVGR,3,HCD,FTMS,5,1710.938663,1.9775,3.4214,True,AAAATVEEPQVGIVGR,16
2,Plamsa_03401_RD2_30min_Top10_86ms_RP13_R1,9832,[UNIMOD:214]AAC[UNIMOD:4]GPSSC[UNIMOD:4]YALFPR,2,HCD,FTMS,7,1699.793263,16.2500,163.6400,False,AACGPSSCYALFPR,14
3,Plamsa_03401_RD2_30min_Top10_86ms_RP13_R1,4680,[UNIMOD:214]AAC[UNIMOD:4]LLPK[UNIMOD:214],2,HCD,FTMS,3,1059.635426,7.9360,91.0930,False,AACLLPK,7
4,Plamsa_03401_RD2_30min_Top10_86ms_RP13_R1,2418,[UNIMOD:214]AAC[UNIMOD:4]LRRC[UNIMOD:4]K[UNIMO...,2,HCD,FTMS,4,1321.731426,4.3071,17.2700,False,AACLRRCK,8
5,Plamsa_03401_RD2_30min_Top10_86ms_RP13_R1,9044,[UNIMOD:214]AAC[UNIMOD:4]LRRC[UNIMOD:4]K[UNIMO...,2,HCD,FTMS,8,1321.731426,15.0040,8.1535,False,AACLRRCK,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15503,Plamsa_03401_RD2_30min_Top10_86ms_RP13_R1,7640,[UNIMOD:214]YYQTDLVNIMK[UNIMOD:214],3,HCD,FTMS,7,1674.889426,12.7810,15.5390,False,YYQTDLVNIMK,11
15504,Plamsa_03401_RD2_30min_Top10_86ms_RP13_R1,10582,[UNIMOD:214]YYSNVC[UNIMOD:4]QR,2,HCD,FTMS,9,1232.572963,17.4690,17.3860,False,YYSNVCQR,8
15505,Plamsa_03401_RD2_30min_Top10_86ms_RP13_R1,15474,[UNIMOD:214]YYTALYRK[UNIMOD:214]MLDPGLMTC[UNIM...,3,HCD,FTMS,6,2742.427289,25.5650,8.8569,False,YYTALYRKMLDPGLMTCSK,19
15506,Plamsa_03401_RD2_30min_Top10_86ms_RP13_R1,12042,[UNIMOD:214]YYVESC[UNIMOD:4]LDQPILQVDK[UNIMOD:...,3,HCD,FTMS,9,2257.154426,19.8670,7.7095,True,YYVESCLDQPILQVDK,16


In [39]:
import re
from typing import List, Optional, Dict
import spectrum_fundamentals.constants as C
MAXQUANT_VAR_MODS = C.MAXQUANT_VAR_MODS
def maxquant_to_internal(sequences: List[str], fixed_mods: Optional[Dict[str, str]] = None) -> List[str]:
    """
    Function to translate a MaxQuant modstring to the Prosit format.

    :param sequences: List[str] of sequences
    :param fixed_mods: Optional dictionary of modifications with key aa and value mod, e.g. 'M': 'M(UNIMOD:35)'.
        Fixed modifications must be included in the variable modificatons dictionary.
        By default, i.e. if nothing is supplied to fixed_mods, carbamidomethylation on cystein will be included
        in the fixed modifications. If you want to have no fixed modifictions at all, supply fixed_mods={}
    :raises AssertionError: if illegal modification was provided in the fixed_mods dictionary.
    :return: a list of modified sequences
    """
    if fixed_mods is None:
        fixed_mods = {"C": "C[UNIMOD:4]"}
    err_msg = f"Provided illegal fixed mod, supported modifications are {set(MAXQUANT_VAR_MODS.values())}."
    assert all(x in MAXQUANT_VAR_MODS.values() for x in fixed_mods.values()), err_msg

    replacements = {**MAXQUANT_VAR_MODS, **fixed_mods}

    def custom_regex_escape(key: str) -> str:
        """
        Subfunction to escape only normal brackets in the modstring.

        :param key: The match to escape
        :return: match with escaped special characters
        """
        for k, v in {"(": r"\(", ")": r"\)"}.items():
            key = key.replace(k, v)
        return key

    regex = re.compile("|".join(map(custom_regex_escape, replacements.keys())))

    def find_replacement(match: re.Match) -> str:
        """
        Subfunction to find the corresponding substitution for a match.

        :param match: an re.Match object found by re.sub
        :return: substitution string for the given match
        """
        key = match.string[match.start() : match.end()]
        if "_" in key:  # If _ is in the match we need to differentiate n and c term
            if match.start() == 0:
                key = f"^{key}"
            else:
                key = f"{key}$"

        return replacements[key]
    #print(sequences)
    return [regex.sub(find_replacement, seq).replace("_", "") for seq in sequences]


In [49]:
import numpy as np
import pandas as pd
tmt_labeled = "itraq4"
unimod_tag = c.TMT_MODS[tmt_labeled]
seq = pd.Series(["_GYEAVWWDKRR_" , "_GVYDPKNYIRDALGFDCK_"]).to_numpy()
fixed_mods={"C": "C[UNIMOD:4]", "^_": f"_{unimod_tag}", "K": f"K{unimod_tag}"}

In [26]:
print(fixed_mods)

{'C': 'C[UNIMOD:4]', '^_': '_[UNIMOD:214]', 'K': 'K[UNIMOD:214]'}


In [50]:
maxquant_to_internal(seq,fixed_mods)

['[UNIMOD:214]GYEAVWWDK[UNIMOD:214]RR',
 '[UNIMOD:214]GVYDPK[UNIMOD:214]NYIRDALGFDC[UNIMOD:4]K[UNIMOD:214]']

In [78]:
data = {'RAW_FILE': ["HEK293_noFAIMS_Fr1.56014.56014.4.0.dta", "HEK293_noFAIMS_Fr1.61150.61150.4.0.dta"]}
df = pd.DataFrame(data)
df['part_1'] = df['RAW_FILE'].str.split('.').str[0]
df['part_2'] = df['RAW_FILE'].str.split('.').str[1]
print(df)
#df['RAW_FILE'].str.split('.')[0][0] 



                                 RAW_FILE              part_1 part_2
0  HEK293_noFAIMS_Fr1.56014.56014.4.0.dta  HEK293_noFAIMS_Fr1  56014
1  HEK293_noFAIMS_Fr1.61150.61150.4.0.dta  HEK293_noFAIMS_Fr1  61150


In [74]:
import pandas as pd

# Create a sample DataFrame
data = {'column': ["HEK293_noFAIMS_Fr1.56014.56014.4.0.dta", "Sample1.12345.67890.1.2.txt"]}
df = pd.DataFrame(data)

# Split the string based on the dot (".") delimiter
df['part_1'] = df['column'].str.split('.').str[0]

# Extract one of the repeated numbers (e.g., the second occurrence)
df['repeated_number'] = df['column'].str.findall(r'\d+').str[1]

# Print the updated DataFrame
print(df)


                                   column              part_1 repeated_number
0  HEK293_noFAIMS_Fr1.56014.56014.4.0.dta  HEK293_noFAIMS_Fr1               1
1             Sample1.12345.67890.1.2.txt             Sample1           12345


In [88]:
import pandas as pd
import re

# Create a sample DataFrame
data = {'crosslink': ["ALAAAGYDVEK(10)-AAKSPAK(3)", "LKEGTYT(5)-DPQSK(2)", "MTQSPAK(7)-KDELF(4)"]}
df = pd.DataFrame(data)

# Extract peptide sequences and crosslink positions using regular expressions
df['peptide_1'] = df['crosslink'].apply(lambda x: re.split(r'[\(\)-]', x)[0])
df['peptide_2'] = df['crosslink'].apply(lambda x: re.split(r'[\(\)-]', x)[3])
df['position_1'] = df['crosslink'].apply(lambda x: re.split(r'[\(\)-]', x)[1])
df['position_2'] = df['crosslink'].apply(lambda x: re.split(r'[\(\)-]', x)[4])
#df['crosslink'].apply(lambda x: re.split(r'[\(\)-]', x))
df

,crosslink,peptide_1,peptide_2,position_1,position_2
0,ALAAAGYDVEK(10)-AAKSPAK(3),ALAAAGYDVEK,AAKSPAK,10,3
1,LKEGTYT(5)-DPQSK(2),LKEGTYT,DPQSK,5,2
2,MTQSPAK(7)-KDELF(4),MTQSPAK,KDELF,7,4


In [97]:
import pandas as pd
import re

# Create a sample DataFrame
data = {'crosslink': ["ALAACCAGYDVEK", "LKCEGTYT", "MTQSPAKC"]}
df = pd.DataFrame(data)
#df["aa"] = df["crosslink"].st.replace("C", "C[UNIMOD:22]")
df['new_column'] = df['crosslink'].str.replace('C', 'C[UNIMOD:122]')

df

,crosslink,new_column
0,ALAACCAGYDVEK,ALAAC[UNIMOD:122]C[UNIMOD:122]AGYDVEK
1,LKCEGTYT,LKC[UNIMOD:122]EGTYT
2,MTQSPAKC,MTQSPAKC[UNIMOD:122]


In [99]:
import pandas as pd

# Create a sample DataFrame
data = {'peptide_sequence': ["ALAAAGYDVEK", "AAKSPAK", "MTQSPAK"], 'position': [10, 3, 7]}
df = pd.DataFrame(data)

# Function to replace 'K' at specific positions with 'K[UNIMOD:22]'
def replace_k_with_unimod(sequence, position):
    for pos in position:
        if pos <= len(sequence):
            sequence = sequence[:pos-1] + 'K[UNIMOD:22]' + sequence[pos:]
    return sequence

# Apply the function to create the modified sequence
df['modified_sequence'] = df.apply(lambda row: replace_k_with_unimod(row['peptide_sequence'], [row['position']]), axis=1)

# Print the updated DataFrame
print(df)


  peptide_sequence  position       modified_sequence
0      ALAAAGYDVEK        10  ALAAAGYDVK[UNIMOD:22]K
1          AAKSPAK         3      AAK[UNIMOD:22]SPAK
2          MTQSPAK         7      MTQSPAK[UNIMOD:22]


In [118]:
import pandas as pd
data = {'peptide_sequence': ["ALAAAGYDVEK", "AAKSPAK", "MTQSPAK"], 'crosslinker_position': [10, 3, 7], 'crosslinker_type': ['DSSO','DSSO','DSSO'] }
df = pd.DataFrame(data)


def add_mod_sequence(peptide_sequence:str, crosslinker_position:int, crosslinker_type:str ):
        """
        Function to add modification in peptide sequnce (eg. replace 'K' with K[UNIMOD:1896] (DSSO)) at the crosslinker_position  '

        :sequence: unmodified peptide sequence
        :position: crosslinker_position
        :return: modified sequence
        """
        if crosslinker_type == "DSSO":
            print("test")
            for pos in crosslinker_position:
                if pos <= len(peptide_sequence):
                    peptide_sequence = peptide_sequence[:pos-1] + 'K[UNIMOD:1896]' + peptide_sequence[pos:]
            return peptide_sequence
            
df['modified_sequence'] = df.apply(lambda row: add_mod_sequence(row['peptide_sequence'], row['crosslinker_position'],row['crosslinker_type']), axis=1)
print(df)


test


TypeError: 'int' object is not iterable

In [119]:
import pandas as pd

data = {
    'peptide_sequence': ["ALAAAGYDVEK", "AAKSPAK", "MTQSPAK"],
    'crosslinker_position': [10, 3, 7],
    'crosslinker_type': ['DSSO', 'DSSO', 'DSSO']
}
df = pd.DataFrame(data)


def add_mod_sequence(peptide_sequence, crosslinker_position, crosslinker_type):
    """
    Function to add modification in peptide sequence (e.g., replace 'K' with K[UNIMOD:1896] (DSSO)) at the crosslinker_position.
    :peptide_sequence: unmodified peptide sequence
    :crosslinker_position: crosslinker_position
    :crosslinker_type: crosslinker_type
    :return: modified sequence
    """
    if crosslinker_type == "DSSO":
        if crosslinker_position <= len(peptide_sequence):
            peptide_sequence = (
                peptide_sequence[:crosslinker_position - 1] +
                'K[UNIMOD:1896]' +
                peptide_sequence[crosslinker_position:]
            )
    return peptide_sequence


df['modified_sequence'] = df.apply(
    lambda row: add_mod_sequence(row['peptide_sequence'], row['crosslinker_position'], row['crosslinker_type']),
    axis=1
)
print(df)


  peptide_sequence  crosslinker_position crosslinker_type  \
0      ALAAAGYDVEK                    10             DSSO   
1          AAKSPAK                     3             DSSO   
2          MTQSPAK                     7             DSSO   

          modified_sequence  
0  ALAAAGYDVK[UNIMOD:1896]K  
1      AAK[UNIMOD:1896]SPAK  
2      MTQSPAK[UNIMOD:1896]  


In [3]:
import pandas as pd

# Example dataframe
data = {
    'Name': ['John', 'Alice', 'Bob', 'David'],
    'Age': [25, 30, 20, 35],
    'Salary': [50000, 60000, 40000, 70000]
}

df = pd.DataFrame(data)

# Create a new column 'Length' representing the length of the 'Name' column
df['Length'] = df['Name'].apply(len)

print(df)


    Name  Age  Salary  Length
0   John   25   50000       4
1  Alice   30   60000       5
2    Bob   20   40000       3
3  David   35   70000       5


In [5]:
import pandas as pd

# Example dataframe
data = {
    'Modifications': ['null', 'Oxidation[M](3);Oxidation[M](7);Oxidation[M](20);Oxidation[M](24)', 'null',"'Oxidation[M](3);Oxidation[M](7)"]
}

df = pd.DataFrame(data)

# Split the Modifications column by semicolon (;)
split_modifications = df['Modifications'].str.split(';')

# Extract modification and position information into separate lists
modifications = []
positions = []
for mods in split_modifications:
    mods_list = []
    pos_list = []
    for mod in mods:
        if mod != 'null':
            mod_name, pos = mod.split('(')
            pos = pos.rstrip(')')
            mods_list.append(mod_name)
            pos_list.append(pos)
    modifications.append(mods_list)
    positions.append(pos_list)

# Create multiple columns for modifications and positions
num_columns = max([len(mods) for mods in modifications])
for i in range(num_columns):
    col_mod = f'modification_{i+1}'
    col_pos = f'position_{i+1}'
    df[col_mod] = [mods[i] if len(mods) > i else None for mods in modifications]
    df[col_pos] = [pos[i] if len(pos) > i else None for pos in positions]

print(df)


                                       Modifications modification_1  \
0                                               null           None   
1  Oxidation[M](3);Oxidation[M](7);Oxidation[M](2...   Oxidation[M]   
2                                               null           None   
3                   'Oxidation[M](3);Oxidation[M](7)  'Oxidation[M]   

  position_1 modification_2 position_2 modification_3 position_3  \
0       None           None       None           None       None   
1          3   Oxidation[M]          7   Oxidation[M]         20   
2       None           None       None           None       None   
3          3   Oxidation[M]          7           None       None   

  modification_4 position_4  
0           None       None  
1   Oxidation[M]         24  
2           None       None  
3           None       None  


In [59]:
import pandas as pd

def extract_modifications(df):
    # Split the column by semicolon (;)
    split_modifications = df['Modifications'].str.split(';')

    # Extract modification and position information into separate lists
    modifications = []
    positions = []
    for mods in split_modifications:
        mods_list = []
        pos_list = []
        for mod in mods:
            if mod != 'null':
                mod_name, pos = mod.split('(')
                pos = pos.rstrip(')')
                mod_name = mod_name.replace('[M]', '').replace('[C]', '')  # Remove [M] and [C]
                mods_list.append(mod_name)
                pos_list.append(pos)
        modifications.append(mods_list)
        positions.append(pos_list)

    # Create multiple columns for modifications and positions
    num_columns = max([len(mods) for mods in modifications])
    for i in range(num_columns):
        col_mod = f'modification_{i+1}'
        col_pos = f'position_{i+1}'
        df[col_mod] = [mods[i] if len(mods) > i else None for mods in modifications]
        df[col_pos] = [pos[i] if len(pos) > i else None for pos in positions]
    
    return df

# Example dataframe
data = {
    'Modifications': ['null', 'Oxidation[M](3);Oxidation[M](7);Oxidation[M](20);Carbamidomethyl[C](2)', 'null' ,"Carbamidomethyl[C](31)"]
}

df = pd.DataFrame(data)

# Apply the function to extract modifications
df = extract_modifications(df)

print(df)


                                       Modifications   modification_1  \
0                                               null             None   
1  Oxidation[M](3);Oxidation[M](7);Oxidation[M](2...        Oxidation   
2                                               null             None   
3                             Carbamidomethyl[C](31)  Carbamidomethyl   

  position_1 modification_2 position_2 modification_3 position_3  \
0       None           None       None           None       None   
1          3      Oxidation          7      Oxidation         20   
2       None           None       None           None       None   
3         31           None       None           None       None   

    modification_4 position_4  
0             None       None  
1  Carbamidomethyl          2  
2             None       None  
3             None       None  


In [64]:
import pandas as pd

# Example DataFrame
data = {
    'peptide_a': ['ABC', 'DEFG', 'HIJKL'],
    'peptide_b': ['MNOP', 'QRST', 'UVWXYZ'],
    'position_1': [4, 8, None],
    'position_2': [7, None, 4],
    'position_3': [10, 12, 11]
}

df = pd.DataFrame(data)

# Get the length of peptide_a
peptide_a_length = len(df['peptide_a'].iloc[0])

# Iterate over columns starting with 'position'
for column in df.columns:
    if column.startswith('position'):
        # Check if the value in the 'position' column is not None
        mask = df[column].notnull()
        # Check if the number in the 'position' column is greater than peptide_a length
        mask_greater_than_length = df[column] > peptide_a_length

        # Update the 'position' column values where conditions are met
        df.loc[mask & mask_greater_than_length, column] -= 3

print(df)


  peptide_a peptide_b  position_1  position_2  position_3
0       ABC      MNOP         1.0         4.0           7
1      DEFG      QRST         5.0         NaN           9
2     HIJKL    UVWXYZ         NaN         1.0           8


In [32]:
data = {
    'peptide_sequence_unmodified_a': ["ALMAAGYDVEK", "AAKSPAK", "MTQDPAK"],
    'peptide_sequence_unmodified_b': ["TMCSIVSYKNNEIIVGDAAK", "DKCGIELIDLR", "KHGMQYK"],
    'modifications': ["Oxidation[M](3);Oxidation[M](16)", "Carbamidomethyl[C](13)", "NULL"],
    'position_crosslinker_a': [11, 3, 7],
    'position_crosslinker_b': [9, 2, 7]
}

df = pd.DataFrame(data)
test = modify_peptide_sequences(df)
print(test)

  peptide_sequence_unmodified_a peptide_sequence_unmodified_b  \
0                   ALMAAGYDVEK          TMCSIVSYKNNEIIVGDAAK   
1                       AAKSPAK                   DKCGIELIDLR   
2                       MTQDPAK                       KHGMQYK   

                      modifications  position_crosslinker_a  \
0  Oxidation[M](3);Oxidation[M](16)                      11   
1            Carbamidomethyl[C](13)                       3   
2                              NULL                       7   

   position_crosslinker_b peptide_sequence_modified_a  \
0                       9                 ALMAAGYDVEK   
1                       2                     AAKSPAK   
2                       7                     MTQDPAK   

  peptide_sequence_modified_b  
0        TMCSIVSYKNNEIIVGDAAK  
1                 DKCGIELIDLR  
2                     KHGMQYK  


In [60]:
import pandas as pd

# Example DataFrame
data = {
    'peptide_a': ['ABC', 'DEFG', 'HIJKL'],
    'peptide_b': ['MNOP', 'QRST', 'UVWXYZ'],
    'position_1': [5, 8, 9],
    'position_2': [7, 6, 4],
    'position_3': [10, 12, 11]
}

df = pd.DataFrame(data)

# Get the length of peptide_a
peptide_a_length = len(df['peptide_a'].iloc[0])

# Iterate over columns starting with 'position'
for column in df.columns:
    if column.startswith('position'):
        # Check if the number in the 'position' column is greater than peptide_a length
        if df[column].iloc[0] > peptide_a_length:
            # Decrease the number in the 'position' column by 3
            df[column] -= 3

print(df)


  peptide_a peptide_b  position_1  position_2  position_3
0       ABC      MNOP           2           4           7
1      DEFG      QRST           5           3           9
2     HIJKL    UVWXYZ           6           1           8


In [67]:
import pandas as pd
import numpy as np

# Example DataFrame
data = {
    'SEQUENCE_A': ["ALKCAGYDVEK", "AKKSMAK", "MTQSPAK"],
    'peptide_a_length': [11, 7, 7],
    'SEQUENCE_B': ["FQMEISQKATR", "VMANSFPDKIK", "VEMCSVWKLKGVWAK"],
    'crosslinker_position_A': [3, 7, 7],
    'crosslinker_position_B': [8, 9, 8],
    'modification_1': ["Carbamidomethyl", 'Oxidation', 'Oxidation'],
    'position_1': [4, 6, 1],
    'modification_2': ["None", 'None', 'Oxidation'],
    'position_2': ["None", "None", 10],
    'modification_3': ["None", 'None', 'Carbamidomethyl'],
    'position_3': ["None", "None", 11]
}

df = pd.DataFrame(data)

# Split characters of SEQUENCE_A and SEQUENCE_B
df['SEQUENCE_A'] = df['SEQUENCE_A'].str.join(' ')
df['SEQUENCE_B'] = df['SEQUENCE_B'].str.join(' ')

# Replace K with K[UNIMOD:1896] at crosslinker_position_A and crosslinker_position_B
df.loc[df['crosslinker_position_A'].notnull(), 'SEQUENCE_A'] = df['SEQUENCE_A'].str.replace('K', 'K[UNIMOD:1896]', n=1)
df.loc[df['crosslinker_position_B'].notnull(), 'SEQUENCE_B'] = df['SEQUENCE_B'].str.replace('K', 'K[UNIMOD:1896]', n=1)

# Iterate over modification columns
for i in range(1, 4):
    mod_col = f'modification_{i}'
    pos_col = f'position_{i}'

    if mod_col in df.columns and pos_col in df.columns:
        # Convert 'None' values to NaN
        df[pos_col] = df[pos_col].replace('None', np.nan)

        # Adjust positions if they are greater than peptide_a_length
        df.loc[df[pos_col] > df['peptide_a_length'], pos_col] -= df['peptide_a_length']

        # Replace characters in SEQUENCE_A based on modification and position
        df.loc[(df[pos_col].notnull()) & (df[mod_col] == 'Carbamidomethyl'), 'SEQUENCE_A'] = df['SEQUENCE_A'].str.slice_replace(
            start=df[pos_col] - 1,
            stop=df[pos_col],
            repl='C[UNIMOD:4]'
        )
        df.loc[(df[pos_col].notnull()) & (df[mod_col] == 'Oxidation'), 'SEQUENCE_A'] = df['SEQUENCE_A'].str.slice_replace(
            start=df[pos_col] - 1,
            stop=df[pos_col],
            repl='M[UNIMOD:35]'
        )

        # Replace characters in SEQUENCE_B based on modification and position
        df.loc[(df[pos_col].notnull()) & (df[mod_col] == 'Carbamidomethyl'), 'SEQUENCE_B'] = df['SEQUENCE_B'].str.slice_replace(
            start=df[pos_col] - 1,
            stop=df[pos_col],
            repl='C[UNIMOD:4]'
        )
        df.loc[(df[pos_col].notnull()) & (df[mod_col] == 'Oxidation'), 'SEQUENCE_B'] = df['SEQUENCE_B'].str.slice_replace(
            start=df[pos_col] - 1,
            stop=df[pos_col],
            repl='M[UNIMOD:35]'
        )

# Remove leading and trailing spaces from SEQUENCE_A and SEQUENCE_B
df['SEQUENCE_A'] = df['SEQUENCE_A'].str.strip()
df['SEQUENCE_B'] = df['SEQUENCE_B'].str.strip()

print(df)


  SEQUENCE_A  peptide_a_length SEQUENCE_B  crosslinker_position_A  \
0        NaN                11        NaN                       3   
1        NaN                 7        NaN                       7   
2        NaN                 7        NaN                       7   

   crosslinker_position_B   modification_1  position_1 modification_2  \
0                       8  Carbamidomethyl           4           None   
1                       9        Oxidation           6           None   
2                       8        Oxidation           1      Oxidation   

   position_2   modification_3  position_3  
0         NaN             None         NaN  
1         NaN             None         NaN  
2         3.0  Carbamidomethyl         4.0  


In [79]:
data = {
    'SEQUENCE_A': ["ALKCAGYDVEK", "AKKSMAK", "MTQSPAK"],
    'peptide_a_length' : [11, 7, 7],
    'SEQUENCE_B': ["FQMEISQKATR", "VMANSFPDKIK", "VEMCSVWKLKGVWAK"],
    'crosslinker_position_A': [3, 7, 7],
    'crosslinker_position_B': [8, 9, 8],
    'modification_1' : ["Carbamidomethyl" , 'Oxidation' ,'Oxidation'],
    'position_1 ' : [4, 6, 1],
    'modification_2' : ["None" , 'None' ,'Oxidation'],
    'position_2 ' : ["None", "None", 10],
    'modification_3' : ["None" , 'None' ,'Carbamidomethyl'],
    'position_3 ' : ["None", "None", 11],
    
}
df = pd.DataFrame(data)
display(df)
"""
def add_mod_sequence(df):
    df['SEQUENCE_A'] = df['SEQUENCE_A'].str.join(' ')
    df['SEQUENCE_B'] = df['SEQUENCE_B'].str.join(' ')
    
    df['SEQUENCE_A'].loc[df['crosslinker_position_A'].notnull()] = df['SEQUENCE_A'].str.replace('K', 'K[UNIMOD:1896]', n=1)
    df['SEQUENCE_B'].loc[df['crosslinker_position_B'].notnull()] = df['SEQUENCE_B'].str.replace('K', 'K[UNIMOD:1896]', n=1)




    
    for column in df.columns:
        if column.startswith('modification'):
            if df[column] == "Carbamidomethyl":
                if df[
            elif df[column] == "Oxidation":   
            
            else:
                break()
            
            
            


#print(df['SEQUENCE_A'])
"""



,SEQUENCE_A,peptide_a_length,SEQUENCE_B,crosslinker_position_A,crosslinker_position_B,modification_1,position_1,modification_2,position_2,modification_3,position_3
0,ALKCAGYDVEK,11,FQMEISQKATR,3,8,Carbamidomethyl,4,None,None,None,None
1,AKKSMAK,7,VMANSFPDKIK,7,9,Oxidation,6,None,None,None,None
2,MTQSPAK,7,VEMCSVWKLKGVWAK,7,8,Oxidation,1,Oxidation,10,Carbamidomethyl,11


'\ndef add_mod_sequence(df):\n    df[\'SEQUENCE_A\'] = df[\'SEQUENCE_A\'].str.join(\' \')\n    df[\'SEQUENCE_B\'] = df[\'SEQUENCE_B\'].str.join(\' \')\n    \n    df[\'SEQUENCE_A\'].loc[df[\'crosslinker_position_A\'].notnull()] = df[\'SEQUENCE_A\'].str.replace(\'K\', \'K[UNIMOD:1896]\', n=1)\n    df[\'SEQUENCE_B\'].loc[df[\'crosslinker_position_B\'].notnull()] = df[\'SEQUENCE_B\'].str.replace(\'K\', \'K[UNIMOD:1896]\', n=1)\n\n\n\n\n    \n    for column in df.columns:\n        if column.startswith(\'modification\'):\n            if df[column] == "Carbamidomethyl":\n                if df[\n            elif df[column] == "Oxidation":   \n            \n            else:\n                break()\n            \n            \n            \n\n\n#print(df[\'SEQUENCE_A\'])\n'

In [59]:
mod = "Carbamidomethyl[C](1)"
seq_a = "CDNTLSKTNFNQYTAAFIPHGNTTR"
seq_b = "LVEIDKLRK"
peptide_A_length = 25
crosslinker_position_a	= 7 
crosslinker_position_b	= 6
mod.split(";")
split_seq_a = [x for x in seq_a]
split_seq_b = [x for x in seq_b]


for mod in mod.split(";"):
    if mod.startswith("Oxidation"):
        modification = "M[UNIMOD:35]"
        pos_mod = int(mod.split("(")[-1][:-1])
        #print(pos_mod)
        if peptide_A_length >= pos_mod :
            split_seq_a[pos_mod-1] = modification
            #print(split_seq_a)
        else:
            pos_mod = pos_mod - peptide_A_length - 3
            split_seq_b[pos_mod-1] = modification
          
    elif mod.startswith("Carbamidomethyl"):
        modification = "C[UNIMOD:4]"
        pos_mod = int(mod.split("(")[-1][:-1])
        if pos_mod <= peptide_A_length :
            split_seq_a[pos_mod-1] = modification
        else:
            pos_mod = pos_mod - peptide_A_length - 3
            split_seq_b[pos_mod-1] = modification
    elif mod == "null":
        break
    else:
        raise AssertionError(f"unknown modification provided:{mod}")

split_seq_a[crosslinker_position_a-1] = "K[UNIMOD:1896]"
split_seq_b[crosslinker_position_b-1] = "K[UNIMOD:1896]"
     
seq_mod_a = ''.join(split_seq_a)    
seq_mod_b = ''.join(split_seq_b)      
print( seq_mod_a) 
print( seq_mod_b) 

C[UNIMOD:4]DNTLSK[UNIMOD:1896]TNFNQYTAAFIPHGNTTR
LVEIDK[UNIMOD:1896]LRK


In [66]:

mod = "Carbamidomethyl[C](1)"
seq_a = "CDNTLSKTNFNQYTAAFIPHGNTTR"
seq_b = "LVEIDKLRK"
peptide_a_length = 25
crosslinker_type = "DSSO"
crosslinker_position_a	= 7 
crosslinker_position_b	= 6
mod.split(";")
split_seq_a = [x for x in seq_a]
split_seq_b = [x for x in seq_b]


for mod in mod.split(";"):
    if mod.startswith("Oxidation"):
        modification = "M[UNIMOD:35]"
        pos_mod = int(mod.split("(")[-1][:-1])
        if peptide_a_length >= pos_mod :
            split_seq_a[pos_mod-1] = modification
        else:
            pos_mod = pos_mod - peptide_a_length - 3
            split_seq_b[pos_mod-1] = modification
        
    elif mod.startswith("Carbamidomethyl"):
        modification = "C[UNIMOD:4]"
        pos_mod = int(mod.split("(")[-1][:-1])
        if pos_mod <= peptide_a_length :
            split_seq_a[pos_mod-1] = modification
        else:
            pos_mod = pos_mod - peptide_a_length - 3
            split_seq_b[pos_mod-1] = modification
    elif mod in ["nan", "null"]:
        break
    else:
        raise AssertionError(f"unknown modification provided:{mod}")
        

if crosslinker_type == "DSSO":
    split_seq_a[crosslinker_position_a-1] = "K[UNIMOD:1896]"
    split_seq_b[crosslinker_position_b-1] = "K[UNIMOD:1896]"
elif crosslinker_type in ["DSBU", "BUURBU"]:
    split_seq_a[crosslinker_position_a-1] = "K[UNIMOD:1884]"
    split_seq_b[crosslinker_position_b-1] = "K[UNIMOD:1884]"
elif crosslinker_type in ["nan", "null"]:
    split_seq_a == split_seq_a
    split_seq_b == split_seq_b                    
else:
    raise AssertionError(f"unknown crosslinker for xl-prosit:{crosslinker_type}")

seq_mod_a = ''.join(split_seq_a)    
seq_mod_b = ''.join(split_seq_b)      

print(seq_mod_a)
print(seq_mod_b)

C[UNIMOD:4]DNTLSK[UNIMOD:1896]TNFNQYTAAFIPHGNTTR
LVEIDK[UNIMOD:1896]LRK


In [4]:
import logging
import os
from abc import abstractmethod
from typing import Optional

import pandas as pd

from spectrum_io.file import csv

logger = logging.getLogger(__name__)


class SearchResults:
    """Handle search results from different software."""

    path: str
    orig_res: pd.DataFrame
    fake_msms: pd.DataFrame

    def __init__(self, path):
        """
        Init Searchresults object.

        :param path: path to file
        """
        self.path = path

    @abstractmethod
    def read_result(self, path: str, tmt_labeled: str):
        """Read result."""
        raise NotImplementedError

    def generate_internal(self, tmt_labeled: str, out_path: Optional[str] = None) -> str:
        """
        Generate df and save to out_path.

        :param out_path: path to output
        :param tmt_labeled: tmt label as str
        :return: path to output file
        """
        if out_path is None:
            out_path = f"{os.path.splitext(self.path)[0]}.prosit"

        if os.path.isfile(out_path):
            logger.info(f"Found search results in internal format at {out_path}, skipping conversion")
            return out_path

        df = self.read_result(self.path, tmt_labeled)
        csv.write_file(df, out_path)

        return out_path

    def read_internal(self, path: str) -> pd.DataFrame:
        """
        Read file from path.

        :param path: path to file
        :return: dataframe after reading the file
        """
        return csv.read_file(path)


ModuleNotFoundError: No module named 'spectrum_io'

In [5]:
import logging
import re
import pandas as pd
import spectrum_fundamentals.constants as c


#from search_results import SearchResults

logger = logging.getLogger(__name__)

class Plink2(SearchResults):
    """Handle search results from Plink2."""

    @staticmethod
    def read_result(path: str, tmt_labeled: str) -> pd.DataFrame:
        """
        Function to read a csv_file and perform some basic formatting.

        :param path: path to csv_file to read
        :return: pd.DataFrame with the formatted data
        """
        logger.info("Reading csv file")
        columns_to_read = ["Title",
                           "Charge",
                          "Precursor_Mass",
                          "Peptide",
                          "Linker",
                          "Modifications",
                          "Score",
                          "LabelID"] 
        df = pd.read_csv(path, usecols=columns_to_read)
        logger.info("Finished reading csv file")

        # Standardize column names
        df = Plink2.update_columns_for_prosit(df)
        df = Plink2.filter_valid_prosit_sequences(df)

        return df

    def add_mod_sequence(seq_a: str,
                         seq_b: str,
                         mod: str,
                         peptide_a_length: int,
                         crosslinker_position_a: int,
                         crosslinker_position_b: int,
                         crosslinker_type: str):
        """
        Function adds modification in peptide sequence for xl-prosit 
        
        :seq_a: unmodified peptide a
        :seq_b: unmodified peptide b
        :mod: all modifications
        :peptide_a_length: lenght of peptide a
        :crosslinker_position_a: crosslinker position of peptide a
        :crosslinker_position_b: crosslinker position of peptide b
        :crosslinker_type: crosslinker tpe eg. DSSO, DSBU
        :return: modified sequence a and b
        """
        split_seq_a = [x for x in seq_a]
        split_seq_b = [x for x in seq_b]

        for mod in mod.split(";"):
            if mod.startswith("Oxidation"):
                modification = "M[UNIMOD:35]"
                pos_mod = int(mod.split("(")[-1][:-1])
                if peptide_a_length >= pos_mod :
                    split_seq_a[pos_mod-1] = modification
                else:
                    pos_mod = pos_mod - peptide_a_length - 3
                    split_seq_b[pos_mod-1] = modification
                
            elif mod.startswith("Carbamidomethyl"):
                modification = "C[UNIMOD:4]"
                pos_mod = int(mod.split("(")[-1][:-1])
                if pos_mod <= peptide_a_length :
                    split_seq_a[pos_mod-1] = modification
                else:
                    pos_mod = pos_mod - peptide_a_length - 3
                    split_seq_b[pos_mod-1] = modification
            elif mod in ["nan", "null"]:
                break
            else:
                raise AssertionError(f"unknown modification provided:{mod}")
                
        crosslinker_type = crosslinker_type.upper()
        if crosslinker_type == "DSSO":
            split_seq_a[crosslinker_position_a-1] = "K[UNIMOD:1896]"
            split_seq_b[crosslinker_position_b-1] = "K[UNIMOD:1896]"
        elif crosslinker_type in ["DSBU", "BUURBU"]:
            split_seq_a[crosslinker_position_a-1] = "K[UNIMOD:1884]"
            split_seq_b[crosslinker_position_b-1] = "K[UNIMOD:1884]"
        elif crosslinker_type in ["nan", "null"]:
            split_seq_a == split_seq_a
            split_seq_b == split_seq_b                    
        else:
            raise AssertionError(f"unknown crosslinker for xl-prosit:{crosslinker_type}")

        seq_mod_a = ''.join(split_seq_a)    
        seq_mod_b = ''.join(split_seq_b)      

        return seq_mod_a, seq_mod_b


    @staticmethod
    def update_columns_for_prosit(df: pd.DataFrame) -> pd.DataFrame:
        """
        Update columns of df to work with Prosit.

        :param df: df to modify
        :return: modified df as pd.DataFrame
        """
        df.rename(columns={"Title": "RAW_FILE", 
                           "Precursor_Mass": "MASS", #Experimental Mass of crosslinked peptides
                           "Charge": "PRECURSOR_CHARGE",
                           "Linker": "CROSSLINKER_TYPE",
                           "Score":"SCORE",
                           "LabelID": "REVERSE"},
                             inplace=True)
        
        if "MASS_ANALYZER" not in df.columns:
            df["MASS_ANALYZER"] = "FTMS"

        if "FRAGMENTATION" not in df.columns:
            df["FRAGMENTATION"] = "HCD"

        df['SCAN_NUMBER'] = df['RAW_FILE'].str.split('.').str[1]
        df['RAW_FILE'] = df['RAW_FILE'].str.split('.').str[0]
        
        logger.info("Converting Plink2 peptide sequence to internal format")
        df['SEQUENCE_A'] = df['Peptide'].apply(lambda x: re.split(r'[\(\)-]', x)[0])
        df['SEQUENCE_B'] = df['Peptide'].apply(lambda x: re.split(r'[\(\)-]', x)[3])
        df['CROSSLINKER_POSITION_A'] = df['Peptide'].apply(lambda x: re.split(r'[\(\)-]', x)[1])
        df['CROSSLINKER_POSITION_B'] = df['Peptide'].apply(lambda x: re.split(r'[\(\)-]', x)[4])

        df["PEPTIDE_LENGTH_A"] = df['SEQUENCE_A'].apply(len) 
        df["PEPTIDE_LENGTH_B"] = df['SEQUENCE_B'].apply(len)

        df['Modifications'] = df['Modifications'].astype('str') 
        df['CROSSLINKER_POSITION_A'] = df['CROSSLINKER_POSITION_A'].astype('int')
        df['CROSSLINKER_POSITION_B'] = df['CROSSLINKER_POSITION_B'].astype('int')
            
        logger.info("Converting MaxQuant peptide sequence to internal format")
        df[['MODIFIED_SEQUENCE_A','MODIFIED_SEQUENCE_B']] = df.apply(lambda row: Plink2.add_mod_sequence(row['SEQUENCE_A'], 
                                                                                 row['SEQUENCE_B'],
                                                                                 row['Modifications'],
                                                                                 row['PEPTIDE_LENGTH_A'],
                                                                                 row['CROSSLINKER_POSITION_A'],
                                                                                 row['CROSSLINKER_POSITION_B'],
                                                                                 row["CROSSLINKER_TYPE"]), axis=1, result_type='expand')
        
        #df["REVERSE"].fillna(False, inplace=True)
        df["REVERSE"].replace(1, False, inplace=True)
        return df

    @staticmethod
    def filter_valid_prosit_sequences(df: pd.DataFrame) -> pd.DataFrame:
        """
        Filter valid Prosit sequences.

        :param df: df to filter
        :return: df after filtration
        """
        logger.info(f"#sequences before filtering for valid prosit sequences: {len(df.index)}")

        df = df[(df["PEPTIDE_LENGTH_A"] <= 30)]
        df = df[df["PEPTIDE_LENGTH_A"] >= 6]
        df = df[(df["PEPTIDE_LENGTH_B"] <= 30)]
        df = df[df["PEPTIDE_LENGTH_B"] >= 6]
        df = df[(~df["SEQUENCE_A"].str.contains("U"))]
        df = df[(~df["SEQUENCE_B"].str.contains("U"))]
        df = df[df["PRECURSOR_CHARGE"] <= 6]
        logger.info(f"#sequences after filtering for valid prosit sequences: {len(df.index)}")

        return df







ModuleNotFoundError: No module named 'spectrum_fundamentals'

In [6]:
path_df = "/home/mkalhor/wilhelmlab/notebooks/msms.csv"
#df = pd.read_csv(path_df)
#display(df)
df = pd.read_csv(path_df)
df_search = Plink2.read_result(path_df,tmt_labeled="")
#df_join = df_search.merge(df_raw, on=["RAW_FILE", "SCAN_NUMBER"])

#df_final.to_csv("/home/mkalhor/wilhelmlab/notebooks/26SProteasome_Scerevisiae_2020.07.24.filtered_cross-linked_spectra_xl_prosit_10.csv")
#display(df)
display(df_search)

NameError: name 'Plink2' is not defined

In [40]:
import pandas as pd

# Create a sample DataFrame
df = pd.DataFrame({'column': [
    'B180815_02_Lumos_LS_IN_130_MycoDSSO_SCX15_hSAX_f31t35_rep1.8059.8059.4.0.dta',
    'B180815_02_Lumos_LS_IN_130_MycoDSSO_SCX15_hSAX_f31t35_rep1.1234.5678.9.0.dta'
]})

# Extract the first number after the first dot using string manipulation and indexing
df['number'] = df['column'].str.split('.').str[1]

# Print the resulting DataFrame
print(df)


                                              column number
0  B180815_02_Lumos_LS_IN_130_MycoDSSO_SCX15_hSAX...   8059
1  B180815_02_Lumos_LS_IN_130_MycoDSSO_SCX15_hSAX...   1234


In [18]:
import subprocess

command = "python oktoberfest/run_oktoberfest.py --search_dir /home/mkalhor/wilhelmlab/notebooks/ --config_path /home/mkalhor/wilhelmlab/notebooks/config.json"
subprocess.run(command, shell=True)


python: can't open file '/home/mkalhor/wilhelmlab/notebooks/oktoberfest/run_oktoberfest.py': [Errno 2] No such file or directory


CompletedProcess(args='python oktoberfest/run_oktoberfest.py --search_dir /home/mkalhor/wilhelmlab/notebooks/ --config_path /home/mkalhor/wilhelmlab/notebooks/config.json', returncode=2)

In [15]:
import pandas as pd

# Creating the first DataFrame
df_search = pd.DataFrame({'RAW_FILE': ['file1', 'file2', 'file3'],
                          'SCAN_NUMBER': [1, 2, 3],
                          'ColumnA': ['A', 'B', 'C']})

# Creating the second DataFrame
df_raw = pd.DataFrame({'RAW_FILE': ['file1', 'file2', 'file3'],
                       'SCAN_NUMBER': [1, 2, 3],
                       'ColumnB': ['X', 'Y', 'Z']})

# Merging on 'RAW_FILE' and 'SCAN_NUMBER' with empty suffixes
df_join = df_search.merge(df_raw, on=["RAW_FILE", "SCAN_NUMBER"], suffixes=('', ''))

print(df_join)


  RAW_FILE  SCAN_NUMBER ColumnA ColumnB
0    file1            1       A       X
1    file2            2       B       Y
2    file3            3       C       Z
